In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data processing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# Traditional ML models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Time series models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Nixtla client for TimeGPT
from nixtla import NixtlaClient

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
df = pd.read_csv('../outputs/data/sales_weather_merged_filled_consolidated.csv')
print(f"Data shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Locations: {df['Location'].unique()}")

Data shape: (286, 18)
Date range: 2019-04-01 to 2024-03-01
Locations: ['Bangalore' 'Chennai' 'Cochin' 'Secunderabad' 'Vijaywada' 'Sricity']


In [3]:
df.groupby('Location')['QTY'].sum()

Location
Bangalore       225400.0
Chennai         783895.0
Cochin          219571.0
Secunderabad    652811.0
Sricity         118356.0
Vijaywada       224564.0
Name: QTY, dtype: float64

In [4]:
df.head()

,Location,QTY,temp_min,temp_max,temp_mean,dwpt_min,dwpt_max,dwpt_mean,rhum_min,rhum_max,rhum_mean,prcp_min,prcp_max,prcp_mean,wspd_min,wspd_max,wspd_mean,date
0,Bangalore,3951.0,19.00,35.250000,27.041071,0.775000,23.275000,15.193029,12.25,100.00,54.193651,0.0,7.766667,0.056111,0.0,36.625,10.347719,2019-05-01
1,Chennai,24969.0,25.10,39.900000,30.982535,17.800000,28.825000,25.123715,32.50,95.75,72.660069,0.0,2.700000,0.019583,0.0,36.575,11.148819,2019-05-01
2,Cochin,3218.0,23.00,35.666667,28.519028,19.233333,27.966667,24.551019,45.00,100.00,80.724537,0.0,10.200000,0.313796,1.0,29.000,7.195787,2019-05-01
3,Secunderabad,17294.0,20.75,39.750000,29.972061,6.650000,25.000000,17.374348,12.00,98.50,50.913699,0.0,3.933333,0.023148,1.7,28.250,10.836726,2019-05-01
4,Bangalore,4709.0,19.60,34.400000,26.270377,10.240000,24.420000,19.395948,24.80,100.00,69.900511,0.0,8.166667,0.159319,0.0,38.960,13.101580,2019-06-01


In [5]:
df_filtered = df[(df['date'] >= '2020-10-01') & df['Location'].isin(['Chennai', 'Cochin', 'Bangalore', 'Secunderabad'])].reset_index(drop=True)

In [6]:
# Create combined dataset by aggregating locations
def create_combined_dataset(df):
    """
    Combine all locations by summing QTY and averaging other features
    """
    print("Creating combined dataset...")
    
    # Group by date and aggregate
    combined_df = df.groupby('date').agg({
        'QTY': 'sum',  # Sum QTY across all locations
        'temp_mean': 'mean',  # Average temperature
        'temp_max': 'mean',   # Average max temperature
        'temp_min': 'mean',   # Average min temperature
        'rhum_mean': 'mean',  # Average humidity
        'wspd_mean': 'mean',  # Average wind speed
        'prcp_mean': 'mean',  # Average precipitation
    }).reset_index()
    return combined_df

# Create the combined dataset
combined_df = create_combined_dataset(df_filtered)
combined_df.head()


Creating combined dataset...


,date,QTY,temp_mean,temp_max,temp_min,rhum_mean,wspd_mean,prcp_mean
0,2020-10-01,21010.0,26.246574,33.0850,21.42625,83.151548,11.501288,0.300712
1,2020-11-01,16952.0,25.998580,32.6375,20.21125,81.203572,8.595730,0.286117
2,2020-12-01,19866.0,24.962569,31.7400,18.61125,82.426068,8.547400,0.303113
3,2021-01-01,35235.0,24.142288,31.4175,16.83750,78.438166,8.741769,0.139533
4,2021-02-01,24029.0,24.382607,32.2400,16.65500,73.167765,8.776383,0.041532


In [7]:
# Import helper functions from previous notebooks
def create_time_features(df):
    """
    Create comprehensive time-based features for forecasting
    """
    df = df.copy()
    
    # Extract temporal indicators
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)
    
    # Cyclical encoding for seasonality
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)
    
    # Weather interaction features
    if 'temp_mean' in df.columns and 'rhum_mean' in df.columns:
        df['temp_humidity_interaction'] = df['temp_mean'] * df['rhum_mean']
    if 'temp_mean' in df.columns and 'wspd_mean' in df.columns:
        df['temp_wind_interaction'] = df['temp_mean'] * df['wspd_mean']
    
    return df

def create_lag_and_rolling_features(df, target_col='QTY', lags=[1, 2, 3, 6, 12], windows=[3, 6, 12]):
    """
    Create lag and rolling features to prevent data leakage
    """
    df = df.copy()
    
    # Create lag features (only using past data)
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
    
    # Create rolling statistics (only using past data)
    for window in windows:
        df[f'{target_col}_rolling_mean_{window}'] = df[target_col].rolling(window=window, min_periods=1).mean().shift(1)
        df[f'{target_col}_rolling_std_{window}'] = df[target_col].rolling(window=window, min_periods=1).std().shift(1)
        df[f'{target_col}_rolling_min_{window}'] = df[target_col].rolling(window=window, min_periods=1).min().shift(1)
        df[f'{target_col}_rolling_max_{window}'] = df[target_col].rolling(window=window, min_periods=1).max().shift(1)
    
    return df

def prepare_data_for_modeling(df, test_months=6):
    """
    Prepare data for modeling with train/test split
    """

    # Create time features
    df = create_time_features(df)
    
    # Create lag and rolling features
    df = create_lag_and_rolling_features(df)
    
    # Split data: last test_months for testing
    split_idx = len(df) - test_months
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    # Remove rows with NaN from lag/rolling features in training set
    train_df = train_df.dropna().reset_index(drop=True)
    
    # Check if we have any training data left
    if len(train_df) == 0:
        print("Warning: No valid training data after feature engineering. Skipping.")
        return None
    
    # Define feature columns (exclude target and non-feature columns)
    exclude_cols = ['QTY', 'date']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    # Prepare features and target
    X_train = train_df[feature_cols].fillna(0)  # Fill any remaining NaN with 0
    y_train = train_df['QTY']
    X_test = test_df[feature_cols].fillna(0)
    y_test = test_df['QTY']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return {
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'train_df': train_df,
        'test_df': test_df,
        'feature_cols': feature_cols,
        'scaler': scaler
    }

def calculate_metrics(y_true, y_pred):
    """
    Calculate evaluation metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

print("Helper functions loaded successfully!")


Helper functions loaded successfully!


In [8]:
# Define model training functions
def create_ml_models():
    """Create a collection of ML models"""
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
        'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1),
        'CatBoost': CatBoostRegressor(iterations=100, random_state=42, verbose=False),
        'SVR': SVR(kernel='rbf'),
        'KNN': KNeighborsRegressor(n_neighbors=5)
    }
    return models

def train_ml_models(X_train, y_train, X_test, y_test):
    """Train all ML models and return predictions"""
    models = create_ml_models()
    results = {}
    
    for name, model in models.items():
        try:
            # Train model
            model.fit(X_train, y_train)
            
            # Make predictions
            y_pred = model.predict(X_test)
            
            # Calculate metrics
            metrics = calculate_metrics(y_test, y_pred)
            results[name] = {
                'model': model,
                'predictions': y_pred,
                'metrics': metrics
            }
            
            print(f"✓ {name}: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
            
        except Exception as e:
            print(f"✗ {name}: Failed - {str(e)}")
    
    return results

def train_arima_model(train_df, test_df):
    """Train ARIMA model"""
    try:
        # Fit ARIMA model
        model = ARIMA(train_df['QTY'], order=(1, 1, 1))
        fitted_model = model.fit()
        
        # Make predictions
        predictions = fitted_model.forecast(steps=len(test_df))
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ ARIMA: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': fitted_model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ ARIMA: Failed - {str(e)}")
        return None

def train_sarima_model(train_df, test_df):
    """Train SARIMA model"""
    try:
        # Fit SARIMA model
        model = SARIMAX(train_df['QTY'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        fitted_model = model.fit(disp=False)
        
        # Make predictions
        predictions = fitted_model.forecast(steps=len(test_df))
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ SARIMA: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': fitted_model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ SARIMA: Failed - {str(e)}")
        return None

def train_prophet_model(train_df, test_df):
    """Train Prophet model"""
    try:
        # Prepare data for Prophet
        prophet_train = train_df[['date', 'QTY']].copy()
        prophet_train.columns = ['ds', 'y']
        
        # Fit Prophet model
        model = Prophet()
        model.fit(prophet_train)
        
        # Make predictions
        future = model.make_future_dataframe(periods=len(test_df))
        forecast = model.predict(future)
        predictions = forecast['yhat'].iloc[-len(test_df):].values
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ Prophet: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ Prophet: Failed - {str(e)}")
        return None

def train_ets_model(train_df, test_df):
    """Train ETS model"""
    try:
        # Fit ETS model
        if len(train_df) < 24:
            # Use simple exponential smoothing if insufficient data
            model = ExponentialSmoothing(train_df['QTY'], seasonal=None)
        else:
            # Use seasonal Holt-Winters
            model = ExponentialSmoothing(train_df['QTY'], seasonal='add', seasonal_periods=12)
        
        fitted_model = model.fit()
        
        # Make predictions
        predictions = fitted_model.forecast(steps=len(test_df))
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ ETS: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': fitted_model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ ETS: Failed - {str(e)}")
        return None

def train_timegpt_model(train_df, test_df):
    """Train TimeGPT model"""
    try:
        # Initialize Nixtla client
        client = NixtlaClient(api_key="nixak-BPWuiu0QLaDocnyGH7oFOutH821mnpHI5jFwgujKGyPGiLCAqNkQGUQ0vp11ZSOXX9msKcsCZgVM8cRu"
)
        
        # Prepare data for TimeGPT
        timegpt_train = train_df[['date', 'QTY']].copy()
        timegpt_train.columns = ['ds', 'y']
        timegpt_train['ds'] = pd.to_datetime(timegpt_train['ds'])
        timegpt_train = timegpt_train.set_index('ds').resample('MS').first().reset_index()
        timegpt_train = timegpt_train.dropna()
        
        if len(timegpt_train) < 12:
            print("✗ TimeGPT: Insufficient data")
            return None
        
        # Make predictions
        predictions = client.forecast(
            df=timegpt_train,
            h=len(test_df),
            freq='MS'
        )
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions['TimeGPT'])
        
        print(f"✓ TimeGPT: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': client,
            'predictions': predictions['TimeGPT'].values,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ TimeGPT: Failed - {str(e)}")
        return None

print("Model training functions defined successfully!")


Model training functions defined successfully!


In [9]:
from sklearn.preprocessing import LabelEncoder
print("\n" + "="*80)
print("TRAINING MODELS")
print("="*80)

# Initialize results storage
individual_results = []

working_df = df_filtered.copy()
working_df['date'] = pd.to_datetime(working_df['date'])

le = LabelEncoder()
working_df['Location'] = le.fit_transform(working_df['Location'])
print(le.classes_)
# Prepare data for location
location_data = prepare_data_for_modeling(working_df, 12)

print(f"Training samples: {len(location_data['y_train'])}")
print(f"Test samples: {len(location_data['y_test'])}")



TRAINING MODELS
['Bangalore' 'Chennai' 'Cochin' 'Secunderabad']
Training samples: 144
Test samples: 12


In [10]:

# Train ML models
print("\n--- Training ML Models ---")
ml_results = train_ml_models(
    location_data['X_train'],
    location_data['y_train'],
    location_data['X_test'],
    location_data['y_test']
)

# Store ML results
for name, result in ml_results.items():
    individual_results.append({
        'Model_Type': 'ML',
        'Model_Name': name,
        'MAE': result['metrics']['MAE'],
        'RMSE': result['metrics']['RMSE'],
        'MAPE': result['metrics']['MAPE'],
        'R2': result['metrics']['R2']
    })

# Train Time Series models
print("\n--- Training Time Series Models ---")

# ARIMA
arima_result = train_arima_model(location_data['train_df'], location_data['test_df'])
if arima_result:
    individual_results.append({
        'Model_Type': 'Time Series',
        'Model_Name': 'ARIMA',
        'MAE': arima_result['metrics']['MAE'],
        'RMSE': arima_result['metrics']['RMSE'],
        'MAPE': arima_result['metrics']['MAPE'],
        'R2': arima_result['metrics']['R2']
    })

# SARIMA
sarima_result = train_sarima_model(location_data['train_df'], location_data['test_df'])
if sarima_result:
    individual_results.append({
        'Model_Type': 'Time Series',
        'Model_Name': 'SARIMA',
        'MAE': sarima_result['metrics']['MAE'],
        'RMSE': sarima_result['metrics']['RMSE'],
        'MAPE': sarima_result['metrics']['MAPE'],
        'R2': sarima_result['metrics']['R2']
    })

# Prophet
prophet_result = train_prophet_model(location_data['train_df'], location_data['test_df'])
if prophet_result:
    individual_results.append({
        'Model_Type': 'Time Series',
        'Model_Name': 'Prophet',
        'MAE': prophet_result['metrics']['MAE'],
        'RMSE': prophet_result['metrics']['RMSE'],
        'MAPE': prophet_result['metrics']['MAPE'],
        'R2': prophet_result['metrics']['R2']
    })

# ETS
ets_result = train_ets_model(location_data['train_df'], location_data['test_df'])
if ets_result:
    individual_results.append({
        'Model_Type': 'Time Series',
        'Model_Name': 'ETS',
        'MAE': ets_result['metrics']['MAE'],
        'RMSE': ets_result['metrics']['RMSE'],
        'MAPE': ets_result['metrics']['MAPE'],
        'R2': ets_result['metrics']['R2']
    })

# TimeGPT
timegpt_result = train_timegpt_model(location_data['train_df'], location_data['test_df'])
if timegpt_result:
    individual_results.append({
        'Model_Type': 'Time Series',
        'Model_Name': 'TimeGPT',
        'MAE': timegpt_result['metrics']['MAE'],
        'RMSE': timegpt_result['metrics']['RMSE'],
        'MAPE': timegpt_result['metrics']['MAPE'],
        'R2': timegpt_result['metrics']['R2']
    })


print(f"\n{'='*80}")
print(f"Individual location analysis completed!")
print(f"Total individual results: {len(individual_results)}")
print(f"{'='*80}")



--- Training ML Models ---
✓ Linear Regression: MAE=5178.62, RMSE=7881.23, R²=0.362
✓ Ridge: MAE=4497.66, RMSE=6914.61, R²=0.509
✓ Lasso: MAE=4792.64, RMSE=7555.39, R²=0.414
✓ ElasticNet: MAE=4429.75, RMSE=6732.91, R²=0.535
✓ Random Forest: MAE=4734.12, RMSE=7482.21, R²=0.425
✓ Gradient Boosting: MAE=4994.51, RMSE=8150.64, R²=0.318
✓ AdaBoost: MAE=4400.00, RMSE=7408.63, R²=0.437
✓ Extra Trees: MAE=5199.23, RMSE=8036.23, R²=0.337
✓ XGBoost: MAE=5135.02, RMSE=8171.69, R²=0.315
✓ LightGBM: MAE=5180.51, RMSE=7815.92, R²=0.373
✓ CatBoost: MAE=5414.14, RMSE=8401.37, R²=0.276
✓ SVR: MAE=8820.43, RMSE=13116.85, R²=-0.766
✓ KNN: MAE=5919.65, RMSE=8933.33, R²=0.181

--- Training Time Series Models ---
✓ ARIMA: MAE=6874.14, RMSE=9669.52, R²=0.040


/Users/sahil/Dev/cdro/rnd/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


✓ SARIMA: MAE=6014.32, RMSE=8812.73, R²=0.203


INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:cmdstanpy:Chain [1] start processing
INFO:cmdstanpy:Chain [1] done processing
INFO:nixtla.nixtla_client:Validating inputs...


✓ Prophet: MAE=7694.29, RMSE=10425.12, R²=-0.116
✓ ETS: MAE=6339.99, RMSE=9303.89, R²=0.112
✗ TimeGPT: Failed - Series contain missing or duplicate timestamps, or the timestamps do not match the provided frequency.
Please make sure that all series have a single observation from the first to the last timestamp and that the provided frequency matches the timestamps'.
You can refer to https://docs.nixtla.io/docs/tutorials-missing_values for an end to end example.

Individual location analysis completed!
Total individual results: 17


In [11]:
# Deep Learning Data Preparation Functions
def prepare_data_for_deep_learning(df, target_col='QTY', sequence_length=12, test_months=6):
    """
    Prepare data for deep learning models with sequence data
    
    Args:
        df: DataFrame with time series data
        target_col: Name of target column
        sequence_length: Number of time steps to look back
        test_months: Number of months to use for testing
    
    Returns:
        Dictionary containing prepared data for deep learning models
    """
    print("Preparing data for deep learning models...")
    
    # Create time features
    df = create_time_features(df)
    
    # Create lag and rolling features
    df = create_lag_and_rolling_features(df, target_col)
    
    # Remove rows with NaN from lag/rolling features
    df = df.dropna().reset_index(drop=True)
    
    if len(df) < sequence_length + test_months:
        print("Warning: Insufficient data for deep learning models")
        return None
    
    # Split data: last test_months for testing
    split_idx = len(df) - test_months
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    # Define feature columns (exclude target and non-feature columns)
    exclude_cols = [target_col, 'date']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    # Prepare features and target
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df[target_col]
    X_test = test_df[feature_cols].fillna(0)
    y_test = test_df[target_col]
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Create sequences for deep learning
    def create_sequences(X, y, seq_length):
        X_seq, y_seq = [], []
        for i in range(seq_length, len(X)):
            X_seq.append(X[i-seq_length:i])
            y_seq.append(y[i])
        return np.array(X_seq), np.array(y_seq)
    
    # Create sequences
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, sequence_length)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test.values, sequence_length)
    
    print(f"Training sequences: {X_train_seq.shape}")
    print(f"Test sequences: {X_test_seq.shape}")
    
    return {
        'X_train': X_train_seq,
        'X_test': X_test_seq,
        'y_train': y_train_seq,
        'y_test': y_test_seq,
        'train_df': train_df,
        'test_df': test_df,
        'feature_cols': feature_cols,
        'scaler': scaler,
        'sequence_length': sequence_length
    }

def create_lstm_model(input_shape, units=50, dropout=0.2):
    """Create LSTM model"""
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=input_shape),
        Dropout(dropout),
        LSTM(units, return_sequences=False),
        Dropout(dropout),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def create_gru_model(input_shape, units=50, dropout=0.2):
    """Create GRU model"""
    model = Sequential([
        GRU(units, return_sequences=True, input_shape=input_shape),
        Dropout(dropout),
        GRU(units, return_sequences=False),
        Dropout(dropout),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def create_cnn_lstm_model(input_shape, filters=64, kernel_size=3, lstm_units=50, dropout=0.2):
    """Create CNN-LSTM hybrid model"""
    from tensorflow.keras.layers import Conv1D, MaxPooling1D
    
    model = Sequential([
        Conv1D(filters=filters, kernel_size=kernel_size, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        LSTM(lstm_units, return_sequences=False),
        Dropout(dropout),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def create_simple_rnn_model(input_shape, units=50, dropout=0.2):
    """Create simple RNN model"""
    model = Sequential([
        tf.keras.layers.SimpleRNN(units, return_sequences=True, input_shape=input_shape),
        Dropout(dropout),
        tf.keras.layers.SimpleRNN(units, return_sequences=False),
        Dropout(dropout),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def train_deep_learning_models(dl_data, epochs=100, batch_size=32, validation_split=0.2):
    """
    Train all deep learning models
    
    Args:
        dl_data: Dictionary containing prepared deep learning data
        epochs: Number of training epochs
        batch_size: Batch size for training
        validation_split: Fraction of data to use for validation
    
    Returns:
        Dictionary containing trained models and results
    """
    print("\n" + "="*60)
    print("TRAINING DEEP LEARNING MODELS")
    print("="*60)
    
    # Set up early stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=15, 
        restore_best_weights=True,
        verbose=1
    )
    
    # Reduce learning rate on plateau
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=0.0001,
        verbose=1
    )
    
    callbacks = [early_stopping, reduce_lr]
    
    # Get input shape
    input_shape = (dl_data['X_train'].shape[1], dl_data['X_train'].shape[2])
    print(f"Input shape: {input_shape}")
    
    # Initialize models
    models = {
        'LSTM': create_lstm_model(input_shape),
        'GRU': create_gru_model(input_shape),
        'CNN-LSTM': create_cnn_lstm_model(input_shape),
        'Simple RNN': create_simple_rnn_model(input_shape)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n--- Training {name} ---")
        try:
            # Train model
            history = model.fit(
                dl_data['X_train'], dl_data['y_train'],
                epochs=epochs,
                batch_size=batch_size,
                validation_split=validation_split,
                callbacks=callbacks,
                verbose=0
            )
            
            # Make predictions
            train_pred = model.predict(dl_data['X_train'], verbose=0)
            test_pred = model.predict(dl_data['X_test'], verbose=0)
            
            # Calculate metrics
            train_metrics = calculate_metrics(dl_data['y_train'], train_pred.flatten())
            test_metrics = calculate_metrics(dl_data['y_test'], test_pred.flatten())
            
            results[name] = {
                'model': model,
                'history': history,
                'train_predictions': train_pred.flatten(),
                'test_predictions': test_pred.flatten(),
                'train_metrics': train_metrics,
                'test_metrics': test_metrics
            }
            
            print(f"✓ {name} - Train MAE: {train_metrics['MAE']:.2f}, Test MAE: {test_metrics['MAE']:.2f}")
            print(f"  Train R²: {train_metrics['R2']:.3f}, Test R²: {test_metrics['R2']:.3f}")
            
        except Exception as e:
            print(f"✗ {name}: Failed - {str(e)}")
            results[name] = None
    
    return results

print("Deep learning functions created successfully!")


Deep learning functions created successfully!


In [12]:
# Prepare data for deep learning models
print("Preparing data for deep learning models...")

# Use the same filtered data but prepare it for deep learning
le=LabelEncoder()
working_df['Location'] = le.fit_transform(working_df['Location'])
dl_data = prepare_data_for_deep_learning(working_df, sequence_length=12, test_months=12)

if dl_data is not None:
    print(f"Deep learning data prepared successfully!")
    print(f"Training sequences: {dl_data['X_train'].shape}")
    print(f"Test sequences: {dl_data['X_test'].shape}")
    print(f"Number of features: {dl_data['X_train'].shape[2]}")
else:
    print("Failed to prepare deep learning data!")


Preparing data for deep learning models...
Preparing data for deep learning models...
Training sequences: (132, 12, 48)
Test sequences: (0,)
Deep learning data prepared successfully!
Training sequences: (132, 12, 48)
Test sequences: (0,)
Number of features: 48


In [ ]:
# Train deep learning models
if dl_data is not None:
    # Train all deep learning models
    dl_results = train_deep_learning_models(dl_data, epochs=50, batch_size=16, validation_split=0.2)
    
    # Store deep learning results
    dl_individual_results = []
    
    for name, result in dl_results.items():
        if result is not None:
            dl_individual_results.append({
                'Model_Type': 'Deep Learning',
                'Model_Name': name,
                'MAE': result['test_metrics']['MAE'],
                'RMSE': result['test_metrics']['RMSE'],
                'MAPE': result['test_metrics']['MAPE'],
                'R2': result['test_metrics']['R2']
            })
    
    print(f"\n{'='*60}")
    print(f"Deep Learning Models Results Summary")
    print(f"{'='*60}")
    
    # Create results DataFrame
    dl_results_df = pd.DataFrame(dl_individual_results)
    print(dl_results_df.to_string(index=False))
    
    # Add to overall results
    individual_results.extend(dl_individual_results)
    
    print(f"\nTotal deep learning models trained: {len(dl_individual_results)}")
    print(f"Total models in analysis: {len(individual_results)}")
else:
    print("Skipping deep learning models due to data preparation failure.")



TRAINING DEEP LEARNING MODELS
Input shape: (12, 48)

--- Training LSTM ---


In [ ]:
# Create comprehensive results comparison
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80)

# Create final results DataFrame
final_results_df = pd.DataFrame(individual_results)

# Sort by R² score (descending)
final_results_df = final_results_df.sort_values('R2', ascending=False)

print("\nTop 10 Models by R² Score:")
print("-" * 50)
top_10 = final_results_df.head(10)[['Model_Type', 'Model_Name', 'MAE', 'RMSE', 'R2']]
print(top_10.to_string(index=False))

print(f"\nTotal models evaluated: {len(final_results_df)}")
print(f"Model types included: {final_results_df['Model_Type'].unique()}")

# Summary by model type
print("\n" + "="*60)
print("SUMMARY BY MODEL TYPE")
print("="*60)

model_type_summary = final_results_df.groupby('Model_Type').agg({
    'MAE': ['mean', 'min'],
    'RMSE': ['mean', 'min'],
    'R2': ['mean', 'max']
}).round(3)

print(model_type_summary)


In [ ]:
# Visualization of Deep Learning Results
if dl_data is not None and 'dl_results' in locals():
    print("\n" + "="*60)
    print("DEEP LEARNING MODEL VISUALIZATIONS")
    print("="*60)
    
    # Plot training history for each model
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, (name, result) in enumerate(dl_results.items()):
        if result is not None and i < 4:
            ax = axes[i]
            
            # Plot training and validation loss
            history = result['history']
            ax.plot(history.history['loss'], label='Training Loss')
            ax.plot(history.history['val_loss'], label='Validation Loss')
            ax.set_title(f'{name} - Training History')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Plot predictions vs actual for test data
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, (name, result) in enumerate(dl_results.items()):
        if result is not None and i < 4:
            ax = axes[i]
            
            # Plot predictions vs actual
            y_test = dl_data['y_test']
            y_pred = result['test_predictions']
            
            ax.scatter(y_test, y_pred, alpha=0.6)
            ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
            ax.set_xlabel('Actual')
            ax.set_ylabel('Predicted')
            ax.set_title(f'{name} - Predictions vs Actual')
            ax.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print("Deep learning visualizations completed!")
else:
    print("No deep learning results available for visualization.")


In [13]:
# OPTIMIZED DEEP LEARNING TRAINING - Faster Settings
print("Stopping current training and restarting with optimized settings...")

# Create optimized deep learning models with smaller architectures
def create_optimized_lstm_model(input_shape, units=32, dropout=0.1):
    """Create optimized LSTM model with smaller architecture"""
    model = Sequential([
        LSTM(units, return_sequences=False, input_shape=input_shape),
        Dropout(dropout),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
    return model

def create_optimized_gru_model(input_shape, units=32, dropout=0.1):
    """Create optimized GRU model with smaller architecture"""
    model = Sequential([
        GRU(units, return_sequences=False, input_shape=input_shape),
        Dropout(dropout),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
    return model

def create_optimized_cnn_lstm_model(input_shape, filters=32, kernel_size=2, lstm_units=32, dropout=0.1):
    """Create optimized CNN-LSTM model"""
    from tensorflow.keras.layers import Conv1D, MaxPooling1D
    
    model = Sequential([
        Conv1D(filters=filters, kernel_size=kernel_size, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        LSTM(lstm_units, return_sequences=False),
        Dropout(dropout),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
    return model

def create_optimized_simple_rnn_model(input_shape, units=32, dropout=0.1):
    """Create optimized Simple RNN model"""
    model = Sequential([
        tf.keras.layers.SimpleRNN(units, return_sequences=False, input_shape=input_shape),
        Dropout(dropout),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
    return model

def train_optimized_deep_learning_models(dl_data, epochs=10, batch_size=64, validation_split=0.2):
    """
    Train optimized deep learning models with faster settings
    """
    print("\n" + "="*60)
    print("TRAINING OPTIMIZED DEEP LEARNING MODELS")
    print("="*60)
    
    # Simplified callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=5, 
        restore_best_weights=True,
        verbose=1
    )
    
    callbacks = [early_stopping]
    
    # Get input shape
    input_shape = (dl_data['X_train'].shape[1], dl_data['X_train'].shape[2])
    print(f"Input shape: {input_shape}")
    
    # Initialize optimized models
    models = {
        'LSTM': create_optimized_lstm_model(input_shape),
        'GRU': create_optimized_gru_model(input_shape),
        'CNN-LSTM': create_optimized_cnn_lstm_model(input_shape),
        'Simple RNN': create_optimized_simple_rnn_model(input_shape)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n--- Training {name} (Optimized) ---")
        try:
            # Train model with faster settings
            history = model.fit(
                dl_data['X_train'], dl_data['y_train'],
                epochs=epochs,
                batch_size=batch_size,
                validation_split=validation_split,
                callbacks=callbacks,
                verbose=1  # Show progress
            )
            
            # Make predictions
            train_pred = model.predict(dl_data['X_train'], verbose=0)
            test_pred = model.predict(dl_data['X_test'], verbose=0)
            
            # Calculate metrics
            train_metrics = calculate_metrics(dl_data['y_train'], train_pred.flatten())
            test_metrics = calculate_metrics(dl_data['y_test'], test_pred.flatten())
            
            results[name] = {
                'model': model,
                'history': history,
                'train_predictions': train_pred.flatten(),
                'test_predictions': test_pred.flatten(),
                'train_metrics': train_metrics,
                'test_metrics': test_metrics
            }
            
            print(f"✓ {name} - Train MAE: {train_metrics['MAE']:.2f}, Test MAE: {test_metrics['MAE']:.2f}")
            print(f"  Train R²: {train_metrics['R2']:.3f}, Test R²: {test_metrics['R2']:.3f}")
            
        except Exception as e:
            print(f"✗ {name}: Failed - {str(e)}")
            results[name] = None
    
    return results

print("Optimized deep learning functions created!")


Stopping current training and restarting with optimized settings...
Optimized deep learning functions created!


In [ ]:
# Run optimized deep learning training
if dl_data is not None:
    print("Starting optimized deep learning training...")
    
    # Train with optimized settings
    dl_results = train_optimized_deep_learning_models(dl_data, epochs=10, batch_size=64, validation_split=0.2)
    
    # Store deep learning results
    dl_individual_results = []
    
    for name, result in dl_results.items():
        if result is not None:
            dl_individual_results.append({
                'Model_Type': 'Deep Learning',
                'Model_Name': name,
                'MAE': result['test_metrics']['MAE'],
                'RMSE': result['test_metrics']['RMSE'],
                'MAPE': result['test_metrics']['MAPE'],
                'R2': result['test_metrics']['R2']
            })
    
    print(f"\n{'='*60}")
    print(f"Optimized Deep Learning Models Results Summary")
    print(f"{'='*60}")
    
    # Create results DataFrame
    dl_results_df = pd.DataFrame(dl_individual_results)
    print(dl_results_df.to_string(index=False))
    
    # Add to overall results
    individual_results.extend(dl_individual_results)
    
    print(f"\nTotal optimized deep learning models trained: {len(dl_individual_results)}")
    print(f"Total models in analysis: {len(individual_results)}")
else:
    print("Skipping deep learning models due to data preparation failure.")


Starting optimized deep learning training...

TRAINING OPTIMIZED DEEP LEARNING MODELS
Input shape: (12, 48)

--- Training LSTM (Optimized) ---
Epoch 1/10


In [14]:
# ULTRA-FAST DEEP LEARNING - Minimal Architecture
print("Creating ultra-fast deep learning models with minimal architecture...")

def create_ultra_fast_lstm_model(input_shape):
    """Create ultra-fast LSTM model with minimal architecture"""
    model = Sequential([
        LSTM(16, input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.1), loss='mse', metrics=['mae'])
    return model

def create_ultra_fast_gru_model(input_shape):
    """Create ultra-fast GRU model with minimal architecture"""
    model = Sequential([
        GRU(16, input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.1), loss='mse', metrics=['mae'])
    return model

def create_ultra_fast_simple_rnn_model(input_shape):
    """Create ultra-fast Simple RNN model"""
    model = Sequential([
        tf.keras.layers.SimpleRNN(16, input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.1), loss='mse', metrics=['mae'])
    return model

def train_ultra_fast_models(dl_data, epochs=5, batch_size=128):
    """
    Train ultra-fast deep learning models
    """
    print("\n" + "="*50)
    print("TRAINING ULTRA-FAST MODELS")
    print("="*50)
    
    # Get input shape
    input_shape = (dl_data['X_train'].shape[1], dl_data['X_train'].shape[2])
    print(f"Input shape: {input_shape}")
    
    # Initialize ultra-fast models
    models = {
        'LSTM': create_ultra_fast_lstm_model(input_shape),
        'GRU': create_ultra_fast_gru_model(input_shape),
        'Simple RNN': create_ultra_fast_simple_rnn_model(input_shape)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n--- Training {name} (Ultra-Fast) ---")
        try:
            # Train model with minimal settings
            history = model.fit(
                dl_data['X_train'], dl_data['y_train'],
                epochs=epochs,
                batch_size=batch_size,
                validation_split=0.1,
                verbose=1
            )
            
            # Make predictions
            train_pred = model.predict(dl_data['X_train'], verbose=0)
            test_pred = model.predict(dl_data['X_test'], verbose=0)
            
            # Calculate metrics
            train_metrics = calculate_metrics(dl_data['y_train'], train_pred.flatten())
            test_metrics = calculate_metrics(dl_data['y_test'], test_pred.flatten())
            
            results[name] = {
                'model': model,
                'history': history,
                'train_predictions': train_pred.flatten(),
                'test_predictions': test_pred.flatten(),
                'train_metrics': train_metrics,
                'test_metrics': test_metrics
            }
            
            print(f"✓ {name} - Train MAE: {train_metrics['MAE']:.2f}, Test MAE: {test_metrics['MAE']:.2f}")
            print(f"  Train R²: {train_metrics['R2']:.3f}, Test R²: {test_metrics['R2']:.3f}")
            
        except Exception as e:
            print(f"✗ {name}: Failed - {str(e)}")
            results[name] = None
    
    return results

print("Ultra-fast deep learning functions created!")


Creating ultra-fast deep learning models with minimal architecture...
Ultra-fast deep learning functions created!


In [ ]:
# Run ultra-fast deep learning training
if dl_data is not None:
    print("Starting ultra-fast deep learning training...")
    
    # Train with ultra-fast settings
    dl_results = train_ultra_fast_models(dl_data, epochs=5, batch_size=128)
    
    # Store deep learning results
    dl_individual_results = []
    
    for name, result in dl_results.items():
        if result is not None:
            dl_individual_results.append({
                'Model_Type': 'Deep Learning',
                'Model_Name': name,
                'MAE': result['test_metrics']['MAE'],
                'RMSE': result['test_metrics']['RMSE'],
                'MAPE': result['test_metrics']['MAPE'],
                'R2': result['test_metrics']['R2']
            })
    
    print(f"\n{'='*60}")
    print(f"Ultra-Fast Deep Learning Models Results Summary")
    print(f"{'='*60}")
    
    # Create results DataFrame
    dl_results_df = pd.DataFrame(dl_individual_results)
    print(dl_results_df.to_string(index=False))
    
    # Add to overall results
    individual_results.extend(dl_individual_results)
    
    print(f"\nTotal ultra-fast deep learning models trained: {len(dl_individual_results)}")
    print(f"Total models in analysis: {len(individual_results)}")
else:
    print("Skipping deep learning models due to data preparation failure.")


Starting ultra-fast deep learning training...

TRAINING ULTRA-FAST MODELS
Input shape: (12, 48)

--- Training LSTM (Ultra-Fast) ---
Epoch 1/5


In [ ]:
# Alternative: Create simplified data for faster training
def prepare_simplified_dl_data(df, target_col='QTY', sequence_length=6, test_months=6):
    """
    Prepare simplified data for deep learning with fewer features
    """
    print("Preparing simplified data for ultra-fast deep learning...")
    
    # Create basic time features only
    df = df.copy()
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    
    # Create only essential lag features
    df[f'{target_col}_lag_1'] = df[target_col].shift(1)
    df[f'{target_col}_lag_2'] = df[target_col].shift(2)
    df[f'{target_col}_lag_3'] = df[target_col].shift(3)
    
    # Remove rows with NaN
    df = df.dropna().reset_index(drop=True)
    
    if len(df) < sequence_length + test_months:
        print("Warning: Insufficient data for deep learning models")
        return None
    
    # Split data
    split_idx = len(df) - test_months
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    # Use only essential features
    essential_features = ['year', 'month', 'quarter', f'{target_col}_lag_1', f'{target_col}_lag_2', f'{target_col}_lag_3']
    
    # Prepare features and target
    X_train = train_df[essential_features].fillna(0)
    y_train = train_df[target_col]
    X_test = test_df[essential_features].fillna(0)
    y_test = test_df[target_col]
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Create sequences
    def create_sequences(X, y, seq_length):
        X_seq, y_seq = [], []
        for i in range(seq_length, len(X)):
            X_seq.append(X[i-seq_length:i])
            y_seq.append(y[i])
        return np.array(X_seq), np.array(y_seq)
    
    # Create sequences
    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, sequence_length)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test.values, sequence_length)
    
    print(f"Simplified training sequences: {X_train_seq.shape}")
    print(f"Simplified test sequences: {X_test_seq.shape}")
    
    return {
        'X_train': X_train_seq,
        'X_test': X_test_seq,
        'y_train': y_train_seq,
        'y_test': y_test_seq,
        'train_df': train_df,
        'test_df': test_df,
        'feature_cols': essential_features,
        'scaler': scaler,
        'sequence_length': sequence_length
    }

# Try with simplified data
print("Creating simplified data for faster training...")
simplified_dl_data = prepare_simplified_dl_data(working_df, sequence_length=6, test_months=12)

if simplified_dl_data is not None:
    print(f"Simplified data prepared successfully!")
    print(f"Training sequences: {simplified_dl_data['X_train'].shape}")
    print(f"Test sequences: {simplified_dl_data['X_test'].shape}")
    print(f"Number of features: {simplified_dl_data['X_train'].shape[2]}")
else:
    print("Failed to prepare simplified deep learning data!")


In [ ]:
# Run ultra-fast training with simplified data
if simplified_dl_data is not None:
    print("Starting ultra-fast training with simplified data...")
    
    # Train with simplified data (much smaller input shape)
    dl_results = train_ultra_fast_models(simplified_dl_data, epochs=3, batch_size=64)
    
    # Store deep learning results
    dl_individual_results = []
    
    for name, result in dl_results.items():
        if result is not None:
            dl_individual_results.append({
                'Model_Type': 'Deep Learning (Simplified)',
                'Model_Name': name,
                'MAE': result['test_metrics']['MAE'],
                'RMSE': result['test_metrics']['RMSE'],
                'MAPE': result['test_metrics']['MAPE'],
                'R2': result['test_metrics']['R2']
            })
    
    print(f"\n{'='*60}")
    print(f"Ultra-Fast Deep Learning Models Results (Simplified Data)")
    print(f"{'='*60}")
    
    # Create results DataFrame
    dl_results_df = pd.DataFrame(dl_individual_results)
    print(dl_results_df.to_string(index=False))
    
    # Add to overall results
    individual_results.extend(dl_individual_results)
    
    print(f"\nTotal ultra-fast deep learning models trained: {len(dl_individual_results)}")
    print(f"Total models in analysis: {len(individual_results)}")
else:
    print("Skipping simplified deep learning models due to data preparation failure.")
